# Power trace info

The archive contains files traces10000.bin and ciphertext10000.bin. These files contain power traces and corresponding ciphertexts. The power traces were measured during AES encryption using 128-bit key implemented in FPGA. The following holds:

## File traces10000.bin
Contains 10,000 power traces. Each power trace consists of 2,000 samples. The file therefore contains 2,000 x 10,000 power samples.
Every power sample is a signed 16-bit integer (int16_t, signed short). The file is therefore 2 x 2,000 x 10,000 = 40,000,000 bytes.
## File ciphertext10000.bin
Contains 10,000 ciphertexts. Each ciphertext consists of 16 bytes. The file therefore contains 16 x 10,000 bytes.
All the data are saved in a "binary format", little-endian (i.e., what you would expect while working on a classic PC (Intel, AMD)).

In every file, first, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the first encryption. After that, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the second encryption. And so on.

In [ ]:
from pathlib import Path
import numpy as np
# 5000 encryptions
# each 2000 traces
# each 16 Byte ciphertexts

data_dir: Path = Path.cwd() / 'ni-hsc-lab-04-data'
ptrace_length = 2000 # samples

ptraces = np.fromfile(data_dir / 'traces10000.bin', dtype=np.int16).reshape(-1, ptrace_length)
print(f'Power traces shape: {ptraces.shape}')

ct_length = 16 # Bytes
ciphertexts = np.fromfile(data_dir / 'ciphertext10000.bin', dtype=np.uint8).reshape(-1, ct_length)
print(f'Ciphertext shape: {ciphertexts.shape}')

assert ptraces.shape[0] == ciphertexts.shape[0]

In [ ]:
ShiftRowInverse = np.array([
    0, 13, 10, 7,   # row 0 → no shift
    4, 1, 14, 11,   # row 1 → shift right by 1
    8, 5, 2, 15,    # row 2 → shift right by 2
    12, 9, 6, 3     # row 3 → shift right by 3
], dtype=np.uint8)

SBoxInverse = np.array(
            [0x52 ,0x09 ,0x6A ,0xD5 ,0x30 ,0x36 ,0xA5 ,0x38 ,0xBF ,0x40 ,0xA3 ,0x9E ,0x81 ,0xF3 ,0xD7 ,0xFB
            ,0x7C ,0xE3 ,0x39 ,0x82 ,0x9B ,0x2F ,0xFF ,0x87 ,0x34 ,0x8E ,0x43 ,0x44 ,0xC4 ,0xDE ,0xE9 ,0xCB
            ,0x54 ,0x7B ,0x94 ,0x32 ,0xA6 ,0xC2 ,0x23 ,0x3D ,0xEE ,0x4C ,0x95 ,0x0B ,0x42 ,0xFA ,0xC3 ,0x4E
            ,0x08 ,0x2E ,0xA1 ,0x66 ,0x28 ,0xD9 ,0x24 ,0xB2 ,0x76 ,0x5B ,0xA2 ,0x49 ,0x6D ,0x8B ,0xD1 ,0x25
            ,0x72 ,0xF8 ,0xF6 ,0x64 ,0x86 ,0x68 ,0x98 ,0x16 ,0xD4 ,0xA4 ,0x5C ,0xCC ,0x5D ,0x65 ,0xB6 ,0x92
            ,0x6C ,0x70 ,0x48 ,0x50 ,0xFD ,0xED ,0xB9 ,0xDA ,0x5E ,0x15 ,0x46 ,0x57 ,0xA7 ,0x8D ,0x9D ,0x84
            ,0x90 ,0xD8 ,0xAB ,0x00 ,0x8C ,0xBC ,0xD3 ,0x0A ,0xF7 ,0xE4 ,0x58 ,0x05 ,0xB8 ,0xB3 ,0x45 ,0x06
            ,0xD0 ,0x2C ,0x1E ,0x8F ,0xCA ,0x3F ,0x0F ,0x02 ,0xC1 ,0xAF ,0xBD ,0x03 ,0x01 ,0x13 ,0x8A ,0x6B
            ,0x3A ,0x91 ,0x11 ,0x41 ,0x4F ,0x67 ,0xDC ,0xEA ,0x97 ,0xF2 ,0xCF ,0xCE ,0xF0 ,0xB4 ,0xE6 ,0x73
            ,0x96 ,0xAC ,0x74 ,0x22 ,0xE7 ,0xAD ,0x35 ,0x85 ,0xE2 ,0xF9 ,0x37 ,0xE8 ,0x1C ,0x75 ,0xDF ,0x6E
            ,0x47 ,0xF1 ,0x1A ,0x71 ,0x1D ,0x29 ,0xC5 ,0x89 ,0x6F ,0xB7 ,0x62 ,0x0E ,0xAA ,0x18 ,0xBE ,0x1B
            ,0xFC ,0x56 ,0x3E ,0x4B ,0xC6 ,0xD2 ,0x79 ,0x20 ,0x9A ,0xDB ,0xC0 ,0xFE ,0x78 ,0xCD ,0x5A ,0xF4
            ,0x1F ,0xDD ,0xA8 ,0x33 ,0x88 ,0x07 ,0xC7 ,0x31 ,0xB1 ,0x12 ,0x10 ,0x59 ,0x27 ,0x80 ,0xEC ,0x5F
            ,0x60 ,0x51 ,0x7F ,0xA9 ,0x19 ,0xB5 ,0x4A ,0x0D ,0x2D ,0xE5 ,0x7A ,0x9F ,0x93 ,0xC9 ,0x9C ,0xEF
            ,0xA0 ,0xE0 ,0x3B ,0x4D ,0xAE ,0x2A ,0xF5 ,0xB0 ,0xC8 ,0xEB ,0xBB ,0x3C ,0x83 ,0x53 ,0x99 ,0x61
            ,0x17 ,0x2B ,0x04 ,0x7E ,0xBA ,0x77 ,0xD6 ,0x26 ,0xE1 ,0x69 ,0x14 ,0x63 ,0x55 ,0x21 ,0x0C ,0x7D],
            dtype=np.uint8)

In [ ]:
def calc_intermediate_values(ct, key_guess, ct_byte_idx):
    """
    :param ct: A single ciphertext.
    :param key_guess: Guess of a single key byte.
    :param ct_byte_idx: Attacked byte indexof the ciphertext.
    :return: Hypothesis of state register after 9. round and state register after 10. round.
    """
    post_round10 = ct[ct_byte_idx]

    # States after given operation. The 10th round in reverse.
    # SR shifts bytes of states, therefore perform the reverse to attack correct corresponding bytes.
    # Ex.: Byte in state register on index 1 after 10th round was byte on index 13 after 9th round.
    post_sub_bytes = ct[ShiftRowInverse[ct_byte_idx]] ^ key_guess
    post_round9 = SBoxInverse[post_sub_bytes]
    return post_round10, post_round9

def calc_leakage_function(v1, v2) -> int:
    """
    Extract 4 LSBs from XOR of intermediate values.
    """
    return 0x000F & (v1 ^ v2)

# Perform Mutual Information analysis attack on a single byte of key using the prepared data

1. Choose a byte of key (subkey) to attack on.
2. Choose following intermediate values: v1: result of the last but one round, v2: ciphertext
3. Choose few (e.g., 4) bits of (v1⊕v2) as leakage function.
4. Compute intermediate values for each of possible subkey values and divide power traces into appropriate number of sets (e.g., 16 sets for 4-bit leakage function).
---
5. Compute (estimate) entropy of the measured data H(O) (at each time point independently).
6. For each value of subkey, compute conditional entropy H(O|L_k=j) for each set (each j); Use them to compute conditional entropy H(O|Lk) (at each time point independently).
7. Compute mutual information I(Lk,O)=H(O)-H(O|Lk).
8. Select the best key candidate as the one with the highest mutual information.

## How to compute entropy H(O)?

For each sampling point i, compute entropy using all power traces. You can implement your own entropy function using the instructions from lecture (histrogram or kernel density estimates), or you can use function provided by your sotfware (e.g., Entropy function in Mathematica).

## How to compute entropy H(O|L=j)?

Exactly the same as H(O), but it is computed using traces in each of j sets (based o L) independently.

## How to compute entropy H(O|L)?

Use the formula from lecture. Entropies H(O|L=j) are calculated in the previous step and the probabilities P(L=j) can be easily computed as ratio between size of each set where P(L=j) and the total number of traces.

In [ ]:
def calc_entropy(data, pdf):
    return - np.sum([pdf(sample) * np.log2(pdf(sample)) for sample in data], axis=0)

def partition_traces(ptraces, ciphertexts, key_guess, key_byte_idx):
    # 16 possible leakage values; 16 = 2^4 where 4 is the # of extracted bits
    trace_sets = [
        [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
    ]

    for i, (ptrace, ct) in enumerate(zip(ptraces, ciphertexts)):
        v1, v2 = calc_intermediate_values(ct, key_guess, key_byte_idx)
        L = calc_leakage_function(v1, v2) # L is an integer 0-15
        trace_sets[L].append(ptrace)

    return trace_sets

In [ ]:
from tqdm import tqdm
from scipy import stats
# attacked byte index
key_byte_idx = 0
n_keys = 256

ptraces = ptraces[:1000,1570:1650]
print(f'Traces shape: {ptraces.shape}')
traces_pdf = stats.gaussian_kde(ptraces.flatten())
print('PDF estimated', flush=True)
H_traces = - np.sum([ traces_pdf(sample) * np.log2(traces_pdf(sample)) for sample in tqdm(ptraces.flatten())])  # traces entropy
print('Trace Entropy estimated', flush=True)

MIs = []

for key_guess in tqdm(range(n_keys)):
    print(f'Calculating mutual information for key guess {hex(key_guess)}')
    partitioning = partition_traces(ptraces, ciphertexts, key_guess, key_byte_idx)

    # estimate conditional entropy for each set; for that we also need to estimate conditional distributuion
    c_entropies = [] # H( O | L_k=j ) ; for all j
    for partition in tqdm(partitioning):
        data = np.stack(partition).flatten()
        partition_pdf = stats.gaussian_kde(data, bw_method='silverman')

        c_entropies.append(
            calc_entropy(data, partition_pdf)  # conditional entropy for given trace set H ( O | L = idx(trace_set) )
        )

    L_pdf = stats.gaussian_kde(np.arange(16))
    c_H_total = - np.sum([ (L_pdf(i) * c_entropies[i]) for i in range(16)]) # total conditional entropy for all partitions

    I = H_traces - c_H_total # mutual information
    MIs.append(I)

print(np.argmax(MIs))

In [ ]:
print(hex(np.argmax(MIs)))

In [ ]:
import matplotlib.pyplot as plt

for trace_set in trace_sets:
        for t_data in np.stack(trace_set).T:
            t_data = np.stack(trace_sets[i]).flatten()

            kde = stats.gaussian_kde(t_data, bw_method='silverman')
            x_range = np.linspace(t_data.min(), t_data.max(), 17)
            pdf_estimate = kde(x_range)

            plt.hist(t_data, bins=17, density=True)
            plt.plot(x_range, pdf_estimate, label='KDE Estimate', color='red', linewidth=2)
            plt.title('Kernel Density Estimation')
            plt.legend()
            plt.show()